<a href="https://colab.research.google.com/github/Kandeta/DataScienceTools/blob/master/cs145_m1_SQL_fa26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 145 -- How to build a Spotify-like data app with SQL


# Learning Goals for SQL

We use a Spotify-like database to learn SQL. Three tables: Users, Songs, Listens.

1. **Basic SQL**: SELECT, FROM, WHERE, DISTINCT, GROUP BY, aggregates, JOINs, LEFT JOINs, ORDER BY, NULL handling, HAVING, set operators (IN, UNION, INTERSECT, EXCEPT).

2. **Modular SQL**: Common Table Expressions (CTEs) and nested queries for computing recommendations.

3. **Intermediate SQL**: Window functions vs GROUP BY for computing song ratings.

4. **Deconstructing SQL**: Analyze LLM-generated queries and debug them step by step.

---

[Optional] Python equivalents for selected queries, to see how SQL maps to loops and dictionaries.


# Section 1: Set up Spotify DB (Basic SQL)


## Setup "schema" and load data

## Setup Instructions
Run cells 4-7 to install Postgres, create the schema, and load the Spotify data. You don't need to read this code.


In [1]:
# --- Setup PostgreSQL in Colab ---
!apt-get -y -qq update > /dev/null 2>&1
!apt-get -y -qq install postgresql > /dev/null 2>&1
!service postgresql start > /dev/null 2>&1
!sudo -u postgres psql -c "CREATE USER colab WITH SUPERUSER PASSWORD 'colab';" > /dev/null 2>&1
!sudo -u postgres psql -c "CREATE DATABASE music_streaming_1;" > /dev/null 2>&1
!pip install -q psycopg2-binary


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 62.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import psycopg2
from IPython.display import display, HTML
DB_CONFIG = {"dbname": "music_streaming_1", "user": "colab", "password": "colab", "host": "localhost", "port": "5432"}
def printSqlResults(cursor, tblName):
    if cursor.description:
        df = pd.DataFrame(cursor.fetchall(), columns=[i[0] for i in cursor.description])
        display(HTML("<b><font color=Green> " + tblName + "</font></b>" + df.to_html(index=False)))
def runSql(caption, query):
    with psycopg2.connect(**DB_CONFIG) as conn:
        conn.autocommit = True
        with conn.cursor() as cursor:
            cursor.execute(query)
            printSqlResults(cursor, caption)




In [ ]:
setup_queries = """
-- Create the Users table
CREATE TABLE IF NOT EXISTS Users (
    user_id SERIAL PRIMARY KEY,      -- Unique identifier for each user
    name VARCHAR(100) NOT NULL,      -- User's full name
    email VARCHAR(100) NOT NULL UNIQUE -- User's email (must be unique)
);

-- Create the Songs table
CREATE TABLE IF NOT EXISTS Songs (
    song_id SERIAL PRIMARY KEY,      -- Unique identifier for each song
    title VARCHAR(100) NOT NULL,     -- Song title
    artist VARCHAR(100) NOT NULL,    -- Artist/Band name
    genre VARCHAR(100)               -- Musical genre
);

-- Create the Listens table
CREATE TABLE IF NOT EXISTS Listens (
    listen_id SERIAL PRIMARY KEY,    -- Unique identifier for each listen event
    user_id INTEGER NOT NULL REFERENCES Users(user_id), -- Link to user
    song_id INTEGER NOT NULL REFERENCES Songs(song_id), -- Link to song
    rating FLOAT,                    -- User's rating (optional)
    listen_time TIMESTAMP            -- When the song was played
);

-- Create the recommendations table
CREATE TABLE IF NOT EXISTS Recommendations (
    user_id INTEGER NOT NULL REFERENCES Users(user_id), -- Link to user
    song_id INTEGER NOT NULL REFERENCES Songs(song_id), -- Link to song
    recommendation_id SERIAL PRIMARY KEY,               -- Unique ID for recommendation
    recommendation_time TIMESTAMP                       -- When rec was generated
);
"""
runSql("Schema Setup", setup_queries)

In [ ]:
# 1. Clear existing data
runSql("Cleanup", "DELETE FROM Listens; DELETE FROM Recommendations; DELETE FROM Songs; DELETE FROM Users;")
# 2. Insert Users
runSql("Insert Users", """
INSERT INTO Users (user_id, name, email)
VALUES
    (1, 'Mickey', 'mickey@example.com'),
    (2, 'Minnie', 'minnie@example.com'),
    (3, 'Daffy', 'daffy@example.com'),
    (4, 'Pluto', 'pluto@example.com');
""")
# 3. Insert Songs
runSql("Insert Songs", """
INSERT INTO Songs (song_id, title, artist, genre)
VALUES
    (1, 'Evermore', 'Taylor Swift', 'Pop'),
    (2, 'Willow', 'Taylor Swift', 'Pop'),
    (3, 'Shape of You', 'Ed Sheeran', 'Rock'),
    (4, 'Photograph', 'Ed Sheeran', 'Rock'),
    (5, 'Shivers', 'Ed Sheeran', 'Rock'),
    (6, 'Yesterday', 'Beatles', 'Classic'),
    (7, 'Yellow Submarine', 'Beatles', 'Classic'),
    (8, 'Hey Jude', 'Beatles', 'Classic'),
    (9, 'Bad Blood', 'Taylor Swift', 'Rock'),
    (10, 'DJ Mix', 'DJ', NULL);
""")
# 4. Insert Listens
runSql("Insert Listens", """
INSERT INTO Listens (listen_id, user_id, song_id, rating, listen_time)
VALUES
    (1, 1, 1, 4.5, '2024-08-30 14:35:00'),
    (2, 1, 2, 4.2, NULL),
    (3, 1, 6, 3.9, '2024-08-29 10:15:00'),
    (4, 2, 2, 4.7, NULL),
    (5, 2, 7, 4.6, '2024-08-28 09:20:00'),
    (6, 2, 8, 3.9, '2024-08-27 16:45:00'),
    (7, 3, 1, 2.9, NULL),
    (8, 3, 2, 4.9, '2024-08-26 12:30:00'),
    (9, 3, 6, NULL, NULL);
""")
# 5. Reset serial sequences (to ensure automatic IDs work for future inserts)
runSql("Sync Sequences", """
SELECT setval('users_user_id_seq', (SELECT MAX(user_id) FROM Users));
SELECT setval('songs_song_id_seq', (SELECT MAX(song_id) FROM Songs));
SELECT setval('listens_listen_id_seq', (SELECT MAX(listen_id) FROM Listens));
""")
# Show results
runSql('Users', "SELECT * FROM Users;")
runSql('Songs', "SELECT * FROM Songs;")
runSql('Listens', "SELECT * FROM Listens;")

setval
9


user_id,name,email
1,Mickey,mickey@example.com
2,Minnie,minnie@example.com
3,Daffy,daffy@example.com
4,Pluto,pluto@example.com


song_id,title,artist,genre
1,Evermore,Taylor Swift,Pop
2,Willow,Taylor Swift,Pop
3,Shape of You,Ed Sheeran,Rock
4,Photograph,Ed Sheeran,Rock
5,Shivers,Ed Sheeran,Rock
6,Yesterday,Beatles,Classic
7,Yellow Submarine,Beatles,Classic
8,Hey Jude,Beatles,Classic
9,Bad Blood,Taylor Swift,Rock
10,DJ Mix,DJ,None


listen_id,user_id,song_id,rating,listen_time
1,1,1,4.5,2024-08-30 14:35:00
2,1,2,4.2,NaT
3,1,6,3.9,2024-08-29 10:15:00
4,2,2,4.7,NaT
5,2,7,4.6,2024-08-28 09:20:00
6,2,8,3.9,2024-08-27 16:45:00
7,3,1,2.9,NaT
8,3,2,4.9,2024-08-26 12:30:00
9,3,6,NaN,NaT


## Basic forms of SELECT, FROM, WHERE, DISTINCT, GROUP BY, Aggregates

### CA Section Walkthrough: SELECT, GROUP BY & JOINs

**Time**: ~20 minutes

We'll run through the examples below. Try to answer each question before running the cell, then verify.

1. **Basic SELECT** (cells below): Run the first few SELECT queries. Pick one and modify the WHERE clause -- change the genre or artist. What changes in the output?

2. **Unsafe GROUP BY**: Look at the unsafe GROUP BY query before running it. Ed Sheeran has 3 Rock songs, Taylor Swift has 1. When you GROUP BY genre, which artist should SQL return for Rock? Run it and see what happens.

3. **JOIN vs LEFT JOIN**: Run the JOIN and LEFT JOIN queries side by side. Which songs appear in the LEFT JOIN result but not in the JOIN result? Why are they missing from the JOIN?

4. **Experiment**: Take the popular songs query and change `JOIN` to `LEFT JOIN`. Run it. What changes in the output? Why?


In [ ]:
"""Goal: Learn basic forms of SELECT, FROM, WHERE, DISTINCT
Three variations on the same query: basic WHERE, AND + LIKE, OR."""

qry_classic_songs = """
-- Classic songs: basic WHERE
SELECT Songs.title, Songs.artist
FROM Songs
WHERE Songs.genre = 'Classic';"""
runSql('Classic songs', qry_classic_songs)

qry_classic2_songs = """
-- Adding AND + LIKE: Classic songs starting with "Ye"
SELECT Songs.title, Songs.artist
FROM Songs
WHERE Songs.genre = 'Classic' AND
      Songs.title LIKE 'Ye%';"""
runSql('Classic songs with LIKE', qry_classic2_songs)

qry_classic3_songs = """
-- Using OR: Classic songs OR songs with song_id < 3
SELECT Songs.title, Songs.artist, Songs.song_id
FROM Songs
WHERE Songs.genre = 'Classic' OR
      Songs.song_id < 3;"""
runSql('Classic OR song_id < 3', qry_classic3_songs)


title,artist
Yesterday,Beatles
Yellow Submarine,Beatles
Hey Jude,Beatles


In [ ]:
qry_genres = """
-- List of all genres in the Songs table
SELECT genre
FROM Songs;"""
runSql('All genres in the Songs table', qry_genres)


genre
Pop
Pop
Rock
Rock
Rock
Classic
Classic
Classic
Rock
None


In [ ]:

qry_distinct = """
-- List of unique genres in the Songs table
SELECT DISTINCT genre
FROM Songs;"""
runSql('Unique genres in the Songs table', qry_distinct)

genre
Pop
Rock
Classic
None


In [ ]:
qry_taylor_count = """
-- Songs by Taylor Swift in different genres
SELECT genre, count(*) as num_songs
FROM Songs
where artist = 'Taylor Swift'
GROUP BY genre;"""
runSql('Count songs by Taylor Swift in different genres', qry_taylor_count)

genre,num_songs
Pop,2
Rock,1


In [ ]:
""" Experiment1 with group by
"""
qry_group_by1 = """
SELECT genre, count(*) as num_songs
FROM Songs
GROUP BY genre;"""
runSql('Count songs by genre', qry_group_by1)

qry_group_by2 = """
SELECT artist, genre, count(*) as num_songs
FROM Songs
GROUP BY artist, genre;"""
runSql('Count songs by genre-artist', qry_group_by2)

genre,num_songs
None,1
Rock,4
Classic,3
Pop,2


artist,genre,num_songs
Ed Sheeran,Rock,3
Taylor Swift,Rock,1
Taylor Swift,Pop,2
Beatles,Classic,3
DJ,None,1


In [ ]:
"""Experiment2 with group by: Unsafe!
Why? Ed Sheeran has 3 songs in Rock and Taylor Swift has 1 in Rock.
When we GROUP BY genre, what artist should you return for Rock genre?
SQL engines will throw an error, or return a random artist in SELECT. Unsafe.
"""

qry_group_by_unsafe = """
SELECT artist, genre, count(*) as num_songs
FROM Songs
GROUP BY genre;"""
runSql('Count songs by genre [unsafe/wrong]', qry_group_by_unsafe)

GroupingError: column "songs.artist" must appear in the GROUP BY clause or be used in an aggregate function
LINE 2: SELECT artist, genre, count(*) as num_songs
               ^


In [ ]:
""" Examples of safer versions of above"""
"""Tip: Always make sure SELECT includes only columns in GROUP BY, or aggregates
of the GROUP BY or extraneous columns (e.g., SUM, COUNT, AVG, etc.)"""

qry_group_by_safe1 = """
SELECT artist, genre, count(*) as num_songs
FROM Songs
GROUP BY artist, genre;"""
runSql('Count songs by artist-genre. Select artists-genre', qry_group_by_safe1)

# Same as qry_group_by_safe1: Just not showing artist.
# You can see the genre, count columns identical to qry_group_by_safe1.
# Takeaway: Query runs the same way, SELECT picks only the output columns
qry_group_by_safe1a = """
SELECT genre, count(*) as num_songs
FROM Songs
GROUP BY artist, genre;"""
runSql('Count songs by artist-genre. Select genre', qry_group_by_safe1a)

qry_group_by_safe2 = """
SELECT genre, count(*) as num_songs
FROM Songs
GROUP BY genre;"""
runSql('Count songs by genre. Select genre', qry_group_by_safe2)

artist,genre,num_songs
Ed Sheeran,Rock,3
Taylor Swift,Rock,1
Taylor Swift,Pop,2
Beatles,Classic,3
DJ,None,1


genre,num_songs
Rock,3
Rock,1
Pop,2
Classic,3
None,1


genre,num_songs
None,1
Rock,4
Classic,3
Pop,2


#### Checkpoint Question #1

Which keyword helps filter **unique** values of a particular column?

*Hint*: Take a closer look at the queries above

## Basic JOINs, LEFT JOINs, ORDER BY

In [ ]:
"""What if we had a single table instead of Users, Songs, Listens?
Do not worry about: Query syntax here. We will also learn about normalizing later.
Focus: Notice
(a) Repeated data: For song_id = 2, "Willow, Taylor Swift" is repeated for
    3 users (Mickey, Minnie and Daffy) listening
(b) Sparse data: No one listened to song_id = 3. A bunch of columns have None.

 """
qry_franken_table = """
SELECT *
FROM Songs
LEFT JOIN Listens ON Songs.song_id = Listens.song_id
LEFT JOIN Users ON Listens.user_id = Users.user_id;
"""
runSql('One big table for SongsListensUsers', qry_franken_table)

song_id,title,artist,genre,listen_id,user_id,song_id,rating,listen_time,user_id,name,email
1,Evermore,Taylor Swift,Pop,1.0,1.0,1.0,4.5,2024-08-30 14:35:00,1.0,Mickey,mickey@example.com
2,Willow,Taylor Swift,Pop,2.0,1.0,2.0,4.2,NaT,1.0,Mickey,mickey@example.com
6,Yesterday,Beatles,Classic,3.0,1.0,6.0,3.9,2024-08-29 10:15:00,1.0,Mickey,mickey@example.com
2,Willow,Taylor Swift,Pop,4.0,2.0,2.0,4.7,NaT,2.0,Minnie,minnie@example.com
7,Yellow Submarine,Beatles,Classic,5.0,2.0,7.0,4.6,2024-08-28 09:20:00,2.0,Minnie,minnie@example.com
8,Hey Jude,Beatles,Classic,6.0,2.0,8.0,3.9,2024-08-27 16:45:00,2.0,Minnie,minnie@example.com
1,Evermore,Taylor Swift,Pop,7.0,3.0,1.0,2.9,NaT,3.0,Daffy,daffy@example.com
2,Willow,Taylor Swift,Pop,8.0,3.0,2.0,4.9,2024-08-26 12:30:00,3.0,Daffy,daffy@example.com
6,Yesterday,Beatles,Classic,9.0,3.0,6.0,NaN,NaT,3.0,Daffy,daffy@example.com
10,DJ Mix,DJ,None,NaN,NaN,NaN,NaN,NaT,NaN,None,None


In [ ]:
""" Goal: Learn basic form of
(a) JOIN for 2 and 3 tables,
(b) GROUP BY for aggregating columns
(c) ORDER BY for sorting """

qry_rating_songs = """
SELECT Songs.song_id, Songs.title, Songs.artist, Listens.rating
FROM Songs
JOIN Listens
ON Songs.song_id = Listens.song_id
WHERE Listens.rating > 4.6;
"""
runSql('Highly rated Songs', qry_rating_songs)

qry_avg_rating_songs = """
SELECT Songs.song_id, Songs.title, Songs.artist, AVG(Listens.rating)
FROM Songs
JOIN Listens
ON Songs.song_id = Listens.song_id
GROUP BY Songs.song_id, Songs.title, Songs.artist
"""
runSql('Average rating for songs', qry_avg_rating_songs)

song_id,title,artist,rating
2,Willow,Taylor Swift,4.7
2,Willow,Taylor Swift,4.9


song_id,title,artist,avg
6,Yesterday,Beatles,3.9
2,Willow,Taylor Swift,4.6
7,Yellow Submarine,Beatles,4.6
1,Evermore,Taylor Swift,3.7
8,Hey Jude,Beatles,3.9


In [ ]:


qry_popular = """
-- Popular songs -- we count the number of listens
SELECT Songs.title, Songs.artist, count(Listens.song_id) as popular
FROM   Songs
JOIN   Listens
ON Songs.song_id = Listens.song_id
GROUP BY Songs.title, Songs.artist
ORDER BY COUNT(Listens.song_id) DESC
LIMIT 10;
"""
runSql('Popular songs', qry_popular)

qry_mickey_taylor_fan = """
SELECT Users.name, Songs.title, Songs.artist
FROM Songs
JOIN Listens ON Songs.song_id = Listens.song_id
JOIN Users ON Listens.user_id = Users.user_id
WHERE Users.name = 'Mickey' AND Songs.artist = 'Taylor Swift';
"""
runSql('Taylors songs listened to by Mickey', qry_mickey_taylor_fan)


title,artist,popular
Willow,Taylor Swift,3
Yesterday,Beatles,2
Evermore,Taylor Swift,2
Yellow Submarine,Beatles,1
Hey Jude,Beatles,1


name,title,artist
Mickey,Evermore,Taylor Swift
Mickey,Willow,Taylor Swift


In [ ]:
""" Goal: Learn basic form of JOIN vs LEFT JOIN
In qry_rating_listened, we only include songs that someone listened to
In qry_rating_all, we see songs (by Ed Sheeran and Taylor's Bad Blood)
included, even though no one listened and rated them.
"""

qry_rating_listened = """
SELECT Songs.song_id, Songs.title, Songs.artist, AVG(Listens.rating)
FROM Songs
JOIN Listens
ON Songs.song_id = Listens.song_id
GROUP BY Songs.song_id, Songs.title, Songs.artist
"""
runSql('Song ratings for songs listened to', qry_rating_listened)

qry_rating_all = """
SELECT Songs.song_id, Songs.title, Songs.artist, AVG(Listens.rating)
FROM Songs
LEFT JOIN Listens
ON Songs.song_id = Listens.song_id
GROUP BY Songs.song_id, Songs.title, artist
"""
runSql('Song ratings for All songs', qry_rating_all)

song_id,title,artist,avg
6,Yesterday,Beatles,3.9
2,Willow,Taylor Swift,4.6
7,Yellow Submarine,Beatles,4.6
1,Evermore,Taylor Swift,3.7
8,Hey Jude,Beatles,3.9


song_id,title,artist,avg
4,Photograph,Ed Sheeran,NaN
10,DJ Mix,DJ,NaN
6,Yesterday,Beatles,3.9
2,Willow,Taylor Swift,4.6
9,Bad Blood,Taylor Swift,NaN
7,Yellow Submarine,Beatles,4.6
3,Shape of You,Ed Sheeran,NaN
1,Evermore,Taylor Swift,3.7
5,Shivers,Ed Sheeran,NaN
8,Hey Jude,Beatles,3.9


#### Checkpoint Question #2

Explain the difference between the keywords `JOIN` and `LEFT JOIN`.

*Hint*: Which `song_id`s are included in the two tables? Which songs are excluded from the first table?

### [Optional] Equivalent python code for Popular Songs and Mickey's Taylor Swift songs

In [ ]:
songs = [
  {"song_id": 1, "title": "Evermore", "artist": "Taylor Swift", "genre": "Pop"},
  {"song_id": 2, "title": "Willow", "artist": "Taylor Swift", "genre": "Pop"},
  {"song_id": 3, "title": "Shape of You", "artist": "Ed Sheeran", "genre": "Rock"},
  {"song_id": 4, "title": "Photograph", "artist": "Ed Sheeran", "genre": "Rock"},
  {"song_id": 5, "title": "Shivers", "artist": "Ed Sheeran", "genre": "Rock"},
  {"song_id": 6, "title": "Yesterday", "artist": "Beatles", "genre": "Classic"},
  {"song_id": 7, "title": "Yellow Submarine", "artist": "Beatles", "genre": "Classic"},
  {"song_id": 8, "title": "Hey Jude", "artist": "Beatles", "genre": "Classic"},
  {"song_id": 9, "title": "Bad Blood", "artist": "Taylor Swift", "genre": "Rock"},
  {"song_id": 10, "title": "DJ Mix", "artist": "DJ", "genre": None}
]

listens = [
  {"listen_id": 1, "user_id": 1, "song_id": 1, "rating": 4.5},
  {"listen_id": 2, "user_id": 1, "song_id": 2, "rating": 4.2},
  {"listen_id": 3, "user_id": 1, "song_id": 6, "rating": 3.9},
  {"listen_id": 4, "user_id": 2, "song_id": 2, "rating": 4.7},
  {"listen_id": 5, "user_id": 2, "song_id": 7, "rating": 4.6},
  {"listen_id": 6, "user_id": 2, "song_id": 8, "rating": 3.9},
  {"listen_id": 7, "user_id": 3, "song_id": 1, "rating": 2.9},
  {"listen_id": 8, "user_id": 3, "song_id": 2, "rating": 4.9},
  {"listen_id": 9, "user_id": 3, "song_id": 6, "rating": None}
]

users = [
  {"user_id": 1, "name": "Mickey", "email": "mickey@example.com"},
  {"user_id": 2, "name": "Minnie", "email": "minnie@example.com"},
  {"user_id": 3, "name": "Daffy", "email": "daffy@example.com"},
  {"user_id": 4, "name": "Pluto", "email": "pluto@example.com"}
]

In [ ]:
"""
Python code to compute popular songs
"""
songs_dict = {}
listens_dict = {}

# INPUT: songs, listens
def compute_popular_songs(songs, listens):
  popular_songs = []

  # Compute the number of listens for each song
  for song in songs:
    listens_count = sum(1 for listen in listens if \
                        listen["song_id"] == song["song_id"])
    popular_songs.append({"title": song["title"],
                          "artist": song["artist"], "popular": listens_count})

  # Sort popular songs by listen count in descending order
  popular_songs.sort(key=lambda x: x["popular"], reverse=True)

  # OUTPUT the top 10 popular songs
  for song in popular_songs[:10]:
    print(f"Title: {song['title']}, Artist: {song['artist']}, \
           Popular: {song['popular']}")

compute_popular_songs(songs, listens)


Title: Willow, Artist: Taylor Swift,            Popular: 3
Title: Evermore, Artist: Taylor Swift,            Popular: 2
Title: Yesterday, Artist: Beatles,            Popular: 2
Title: Yellow Submarine, Artist: Beatles,            Popular: 1
Title: Hey Jude, Artist: Beatles,            Popular: 1
Title: Shape of You, Artist: Ed Sheeran,            Popular: 0
Title: Photograph, Artist: Ed Sheeran,            Popular: 0
Title: Shivers, Artist: Ed Sheeran,            Popular: 0
Title: Bad Blood, Artist: Taylor Swift,            Popular: 0
Title: DJ Mix, Artist: DJ,            Popular: 0


In [ ]:
# For Minnie's Taylor Swift songs
results = []

# Perform the join and filter operation
for listen in listens:
  user_id = listen["user_id"]
  song_id = listen["song_id"]

  # Find the corresponding user and song dictionaries
  user = next((user for user in users if user["user_id"] == user_id), None)
  song = next((song for song in songs if song["song_id"] == song_id), None)

  # If both user and song are found, check for the conditions
  if user and song and user["name"] == "Mickey" and song["artist"] == "Taylor Swift":
    results.append({"name": user["name"], "title": song["title"],
                    "artist": song["artist"]})

# Print the results
for result in results:
  print(f"Name: {result['name']}, Title: {result['title']}, Artist: {result['artist']}")

Name: Mickey, Title: Evermore, Artist: Taylor Swift
Name: Mickey, Title: Willow, Artist: Taylor Swift


In [ ]:
# For JOIN vs LEFT JOIN example
from collections import defaultdict

def get_song_ratings_listened_to(songs, listens):
  song_ratings = defaultdict(list)

  # Compute song ratings for songs that have been listened to
  for listen in listens:
    song_id = listen["song_id"]
    rating = listen["rating"]

    if song_id in song_ratings and rating is not None:
      song_ratings[song_id].append(rating)
    elif rating is not None:
      song_ratings[song_id] = [rating]
  results = []
  print(f'{song_ratings=}')
  # Calculate average rating for each song
  for song in songs:
    song_id = song["song_id"]
    title = song["title"]
    artist = song["artist"]

    if song_id in song_ratings:
      ratings = song_ratings.get(song_id, [])
      avg_rating = sum(ratings) / len(ratings) if ratings else None
      results.append({"song_id": song_id, "title": title,
                      "artist": artist, "avg_rating": avg_rating})

  for song_rating in results:
    print(song_rating)
  return results

def get_song_ratings_all(songs, listens):
  song_ratings = defaultdict(list)

  # Compute song ratings for all songs
  for listen in listens:
    song_id = listen["song_id"]
    rating = listen["rating"]

    if rating is not None:
      song_ratings[song_id].append(rating)
  print(f'{song_ratings=}')
  # Include songs that have not been listened to
  for song in songs:
    song_id = song["song_id"]
    if song_id not in song_ratings:
      song_ratings[song_id] = []

  results = []

  # Calculate average rating for each song
  for song in songs:
    song_id = song["song_id"]
    title = song["title"]
    artist = song["artist"]

    ratings = song_ratings[song_id]
    avg_rating = sum(ratings) / len(ratings) if ratings else None
    results.append({"song_id": song_id, "title": title, "artist": artist,
                    "avg_rating": avg_rating})

  for song_rating in results:
    print(song_rating)
  return results

print(f'Listened to songs')
song_ratings_listened_to = get_song_ratings_listened_to(songs, listens)
print(f'For all songs')
song_ratings_all = get_song_ratings_all(songs, listens)


Listened to songs
song_ratings=defaultdict(<class 'list'>, {1: [4.5, 2.9], 2: [4.2, 4.7, 4.9], 6: [3.9], 7: [4.6], 8: [3.9]})
{'song_id': 1, 'title': 'Evermore', 'artist': 'Taylor Swift', 'avg_rating': 3.7}
{'song_id': 2, 'title': 'Willow', 'artist': 'Taylor Swift', 'avg_rating': 4.6000000000000005}
{'song_id': 6, 'title': 'Yesterday', 'artist': 'Beatles', 'avg_rating': 3.9}
{'song_id': 7, 'title': 'Yellow Submarine', 'artist': 'Beatles', 'avg_rating': 4.6}
{'song_id': 8, 'title': 'Hey Jude', 'artist': 'Beatles', 'avg_rating': 3.9}
For all songs
song_ratings=defaultdict(<class 'list'>, {1: [4.5, 2.9], 2: [4.2, 4.7, 4.9], 6: [3.9], 7: [4.6], 8: [3.9]})
{'song_id': 1, 'title': 'Evermore', 'artist': 'Taylor Swift', 'avg_rating': 3.7}
{'song_id': 2, 'title': 'Willow', 'artist': 'Taylor Swift', 'avg_rating': 4.6000000000000005}
{'song_id': 3, 'title': 'Shape of You', 'artist': 'Ed Sheeran', 'avg_rating': None}
{'song_id': 4, 'title': 'Photograph', 'artist': 'Ed Sheeran', 'avg_rating': None}

## Basic SQL queries (NULL values)

### CA Section Walkthrough: NULLs, Set Operators & HAVING

**Time**: ~20 minutes

NULL values and aggregate filtering are where most SQL bugs hide. We'll trace through the behavior carefully.

1. **NULL in aggregates**: Run the Beatles ratings query. One rating is NULL. Before running the COUNT/AVG query, predict: does COUNT include the NULL row? Does AVG? Run it and check.

2. **WHERE vs HAVING**: Run both queries. Both filter rows, but at different stages. Why can't you replace HAVING with WHERE here? What would go wrong?

3. **UNION vs UNION ALL**: Run both. How many rows differ between the two results? Why does UNION remove some rows that UNION ALL keeps?

4. **INTERSECT and EXCEPT**: Before running the EXCEPT query, predict which artists will appear. Run it. Then swap the two SELECTs in the EXCEPT -- does the result change?


In [ ]:
qry_beatles_ratings = """
-- Handling NULL values
SELECT user_id, l.song_id, rating
FROM Listens l
JOIN Songs s
ON l.song_id = s.song_id
WHERE s.artist = 'Beatles';"""
runSql('Ratings for Beatles songs', qry_beatles_ratings)

qry_beatles_ratings_non_null = """
-- Handling NULL values
SELECT user_id, l.song_id, rating
FROM Listens l
JOIN Songs s
ON l.song_id = s.song_id
WHERE s.artist = 'Beatles' AND
      l.rating is NOT NULL;"""
runSql('Ratings for Beatles songs', qry_beatles_ratings_non_null)

user_id,song_id,rating
1,6,3.9
2,7,4.6
2,8,3.9
3,6,NaN


user_id,song_id,rating
1,6,3.9
2,7,4.6
2,8,3.9


In [ ]:

# Note: In Listens, Beatles' Yesterday (userid=3, songid=6)
# has NULL rating. See how NULL ratings are treated
# while counting and taking averages.
qry_count_non_null_ratings = """
SELECT COUNT(rating) AS non_null_ratings
FROM Listens l
JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist = 'Beatles';"""
runSql('Count ratings1', qry_count_non_null_ratings)

qry_count_include_null_ratings = """
SELECT COUNT(*) AS include_null_ratings
FROM Listens l
JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist = 'Beatles';"""
runSql('Count ratings2', qry_count_include_null_ratings)

qry_avg_rating = """
SELECT AVG(rating) AS average_rating
FROM Listens l
JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist = 'Beatles';"""
runSql('Average rating', qry_avg_rating)

qry_avg_rating_non_null = """
SELECT AVG(rating) AS average_rating
FROM Listens l
JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist = 'Beatles' AND
      l.rating is NOT NULL;"""
runSql('Average rating', qry_avg_rating_non_null)

non_null_ratings
3


include_null_ratings
4


average_rating
4.133333


average_rating
4.133333


#### Checkpoint Question #3

Which of the following conditional statements can filter out `NULL` values within a column when placed in a `WHERE` clause? Justify your answers.

(a) `l.rating IS NOT NULL`

(b) `l.rating IS != NULL`

(c) `l.rating <> NULL`

(d) `l.rating != l.rating`

## HAVING on Aggregates vs WHERE

In [ ]:
"""Goal: Learn difference between WHERE and HAVING"""

whereQuery = """
SELECT Listens.user_id, Listens.song_id, rating
FROM Listens
WHERE Listens.user_id = 1"""
runSql('WHERE filters individual rows', whereQuery)


user_id,song_id,rating
1,1,4.5
1,2,4.2
1,6,3.9


In [ ]:
groupQuery1 = """
SELECT Listens.user_id, AVG(rating)
FROM Listens
GROUP BY Listens.user_id"""
runSql(f'Intro query', groupQuery1)

havingQuery = """
SELECT Listens.user_id, AVG(rating)
FROM Listens
GROUP BY Listens.user_id
HAVING AVG(rating) > 4"""
runSql(f'HAVING query', havingQuery)

user_id,avg
3,3.9
2,4.4
1,4.2


user_id,avg
2,4.4
1,4.2


## Set Operators

In [ ]:
""" Goal: Learn basic forms of IN, NOT
IN: Checks if a value is in a set of values or rows
NOT: Negates a condition"""
qry_songs_by = """
--titles and artists of songs by either Ed Sheeran or Taylor Swift.
SELECT title, artist
FROM Songs
WHERE artist IN ('Ed Sheeran', 'Taylor Swift');
"""
runSql('Songs by Ed and Taylor', qry_songs_by)

qry_songs_by = """
--titles and artists of songs by neither Ed Sheeran or Taylor Swift.
SELECT title, artist
FROM Songs
WHERE artist NOT IN ('Ed Sheeran', 'Taylor Swift');
"""
runSql('Songs not by Ed and Taylor', qry_songs_by)

title,artist
Evermore,Taylor Swift
Willow,Taylor Swift
Shape of You,Ed Sheeran
Photograph,Ed Sheeran
Shivers,Ed Sheeran
Bad Blood,Taylor Swift


title,artist
Yesterday,Beatles
Yellow Submarine,Beatles
Hey Jude,Beatles
DJ Mix,DJ


In [ ]:
""" Goal: Learn basic forms of UNION, UNION ALL"""
qry_union = """
-- Titles and artists of songs from both Pop and Rock genres
SELECT title, artist
FROM Songs
WHERE genre = 'Pop'
UNION
SELECT title, artist
FROM Songs
WHERE genre = 'Rock';
"""
runSql('Songs from Pop and Rock genres', qry_union)

qry_union_artists = """
-- Artists of songs from both Pop and Rock genres
SELECT artist
FROM Songs
WHERE genre = 'Pop'
UNION
SELECT artist
FROM Songs
WHERE genre = 'Rock';
"""
runSql('Artists from Pop or Rock genres', qry_union_artists)

qry_union_all = """
-- Titles and artists of songs from both Pop and Rock genres, including duplicates
SELECT artist
FROM Songs
WHERE genre = 'Pop'
UNION ALL
SELECT artist
FROM Songs
WHERE genre = 'Rock';
"""
runSql('All artists from Pop and Rock genres (with duplicates)', qry_union_all)

title,artist
Bad Blood,Taylor Swift
Evermore,Taylor Swift
Photograph,Ed Sheeran
Shape of You,Ed Sheeran
Shivers,Ed Sheeran
Willow,Taylor Swift


artist
Ed Sheeran
Taylor Swift


artist
Taylor Swift
Taylor Swift
Ed Sheeran
Ed Sheeran
Ed Sheeran
Taylor Swift


In [ ]:
""" Goal: Learn basic forms of INTERSECT, EXCEPT
Note: INTERSECT ALL is not in SQL standard. Some DBs like Oracle
support it, and retain duplicates. Always check the doc of a DB for
extra operators not in SQL standard.
"""
qry_intersect = """
-- Artists with songs that belong to both Pop and Rock genres
SELECT artist
FROM Songs
WHERE genre = 'Pop'
INTERSECT
SELECT artist
FROM Songs
WHERE genre = 'Rock';
"""
runSql('Songs in both Pop and Rock genres', qry_intersect)

qry_except = """
-- Artists with songs in Rock genre, but none in the Pop genre
SELECT artist
FROM Songs
WHERE genre = 'Rock'
EXCEPT
SELECT artist
FROM Songs
WHERE genre = 'Pop';
"""
runSql('Songs in Rock but not in Pop genre', qry_except)


artist
Taylor Swift


artist
Ed Sheeran


## Sub-queries

### CA Section Walkthrough: Sub-queries, CTEs & Window Functions

**Time**: ~20 minutes

Sub-queries and CTEs are how you build complex queries from simple pieces. Window functions let you compute aggregates without collapsing rows.

1. **IN subquery**: Run the query that finds songs listened to by user_id=1. Could you rewrite this as a JOIN instead? Try it -- do you get the same result?

2. **IN vs EXISTS**: Run both versions. Same output? EXISTS stops as soon as it finds one match -- why is that cheaper than IN, which computes the full set?

3. **Correlated subquery**: Look at the "better rated in genre" query. Pick one row and trace through it by hand. For a Pop song, what does the inner query return? How does the outer query use that value?

4. **CTEs**: Run the recommendation query. Now run just the `song_similarity` CTE by itself (copy it out, remove the outer query). What does the intermediate table look like?

5. **Window functions**: Run the GROUP BY and WINDOW versions side by side. GROUP BY collapses rows. The WINDOW version keeps every row but adds the aggregate as a new column. When would you want one vs. the other?


In [ ]:
""" Goal: Learn basic forms of sub-queries
Sub-queries: Queries within queries
"""

qry_listens_by_userid = """
-- titles and artists of songs that have been listened to by user_id = 1).
SELECT title, artist
FROM Songs
WHERE song_id IN (SELECT song_id FROM Listens WHERE user_id = 1);
"""
runSql('Songs listened to by user_id=1', qry_listens_by_userid )

qry_unlistened_songs = """
-- Retrieve songs that have not been listened to by user with ID 1
SELECT *
FROM Songs
WHERE song_id NOT IN (
SELECT song_id
FROM Listens
WHERE user_id = 1
);"""
runSql('Unlistened Songs', qry_unlistened_songs)

qry_unlistened_songs = """
-- Retrieve Pop songs that have been listened to by user with ID 1
SELECT *
FROM Songs
WHERE song_id IN (
SELECT song_id
FROM Listens
WHERE user_id = 1 and Songs.genre = 'Pop'
);"""
runSql('Pop Songs listened by user 1', qry_unlistened_songs)

title,artist
Evermore,Taylor Swift
Willow,Taylor Swift
Yesterday,Beatles


song_id,title,artist,genre
3,Shape of You,Ed Sheeran,Rock
4,Photograph,Ed Sheeran,Rock
5,Shivers,Ed Sheeran,Rock
7,Yellow Submarine,Beatles,Classic
8,Hey Jude,Beatles,Classic
9,Bad Blood,Taylor Swift,Rock
10,DJ Mix,DJ,None


song_id,title,artist,genre
1,Evermore,Taylor Swift,Pop
2,Willow,Taylor Swift,Pop


In [ ]:
"""IN vs EXISTS: both find songs that someone listened to.
Same result, different approach. EXISTS stops at the first match -- often cheaper."""

qry_listened_IN = """
-- Using IN: compute the full set, then check membership
SELECT Songs.title, Songs.artist
FROM Songs
WHERE Songs.song_id IN (
  SELECT Listens.song_id FROM Listens
);
"""
runSql('Songs listened to (IN)', qry_listened_IN)

qry_listened_EXISTS = """
-- Using EXISTS: stop as soon as one match is found
SELECT Songs.title, Songs.artist
FROM Songs
WHERE EXISTS (
  SELECT 1 FROM Listens
  WHERE Listens.song_id = Songs.song_id
);
"""
runSql('Songs listened to (EXISTS)', qry_listened_EXISTS)


title,artist
Evermore,Taylor Swift
Willow,Taylor Swift
Yesterday,Beatles
Yellow Submarine,Beatles
Hey Jude,Beatles


### Correlated Subqueries

A correlated subquery runs once *per row* of the outer query. The inner query references the outer row.


In [ ]:
qry_TS_better_rated_ingenre = """
-- Songs by Taylor Swift with an avg-rating higher than
-- the avg-rating of songs in the same genre
SELECT distinct S1.title, S1.genre, AVG(L1.rating)
FROM Listens L1
JOIN Songs S1 ON L1.song_id = S1.song_id
WHERE S1.artist = 'Taylor Swift'
-- Group to calculate the avg-rating for each song
GROUP BY S1.song_id, S1.genre, S1.title
HAVING AVG(L1.rating) > (
    -- Subquery: L2.rating is avg-rating of all songs in S1.genre
    SELECT AVG(L2.rating)
    FROM Listens L2
    JOIN Songs S2 ON L2.song_id = S2.song_id
    WHERE S2.genre = S1.genre
);
"""
runSql('Songs by TS higher rated than average in their genre',
       qry_TS_better_rated_ingenre)


title,genre,avg
Willow,Pop,4.6


In [ ]:
qry_better_rated_ingenre = """
-- Songs with an avg-rating higher than
-- the avg-rating of songs in the same genre
SELECT distinct S1.artist, S1.title, S1.genre, AVG(L1.rating)
FROM Listens L1
JOIN Songs S1 ON L1.song_id = S1.song_id
-- Group to calculate the average rating for each song
GROUP BY S1.song_id, S1.title, S1.artist, S1.genre
HAVING AVG(L1.rating) > (
    -- Subquery: L2.rating is average rating of all songs in S1.genre
    SELECT AVG(L2.rating)
    FROM Listens L2
    JOIN Songs S2 ON L2.song_id = S2.song_id
    WHERE S2.genre = S1.genre
);
"""
runSql('Songs higher rated than average in their genre', qry_better_rated_ingenre)


artist,title,genre,avg
Beatles,Yellow Submarine,Classic,4.6
Taylor Swift,Willow,Pop,4.6


In [ ]:
qry_better_rated_ingenre_all = """
-- Songs by TS with an average rating >= than
-- all average ratings of other songs in the same genre
--
SELECT distinct S1.title
FROM Listens L1
JOIN Songs S1 ON L1.song_id = S1.song_id
WHERE S1.artist = 'Taylor Swift'
-- Group to calculate the average rating for each song
GROUP BY S1.song_id, S1.title
HAVING AVG(L1.rating) >= ALL (
    -- Subquery: L2.rating is average rating of each song in S1.genre
    SELECT AVG(L2.rating)
    FROM Listens L2
    JOIN Songs S2 ON L2.song_id = S2.song_id
    WHERE S2.genre = S1.genre
    GROUP BY L2.song_id
);
"""
runSql('Songs by TS better rated than all others in their genre', qry_better_rated_ingenre_all)




title
Willow


# Section 2: Computing recommendations (Intermediate)
**Common Table Expressions for modular code**

In [ ]:
""" Goal: Learn basic forms of
- Common Table Expressions (CTEs) for modular code"""

# Our dataset is small, so we use count(*) > 1.
# In production with millions of listens, you'd set a higher threshold (e.g., 10000).
similarityQuery = """
WITH song_similarity AS (
    SELECT u1.song_id as song1, u2.song_id as song2, COUNT(*) as common_users
    FROM Listens u1
    JOIN Listens u2
    ON u1.user_id = u2.user_id
    AND u1.song_id != u2.song_id
    GROUP BY u1.song_id, u2.song_id
    -- For this example, looking for song pairs shared across > 1 user
    HAVING count(*) > 1),

-- CTE for recs. For each user, find more songs related to what they listen.
    recs AS (
    SELECT user_id, song2 as song_id
    FROM song_similarity
    JOIN Listens as L
    ON L.song_id = song_similarity.song1
    -- only insert a song, if it's 'new' for the user (not in user's Listens)
    WHERE song_similarity.song2 not in
      (select song_id from listens as tmpL where tmpL.user_id = L.user_id))

-- ### Debug: If you want to check what these CTEs produce, uncomment one of them.
select * from song_similarity
--select * from recs;

-- Insert into recommendations table for later usage
--INSERT INTO Recommendations (user_id, song_id, recommendation_id, recommendation_time)
--SELECT user_id, song_id,
--       randomblob(4) as recommendation_id,
--       datetime('now') as recommendation_time
--FROM recs;
"""

# For writes, use below pattern. We'll learn about Transactions and Commits later.
runSql('Recommendations', similarityQuery)

song1,song2,common_users
6,1,2
1,2,2
1,6,2
2,1,2
6,2,2
2,6,2


In [ ]:
runSql("Recommendations", """SELECT * from Recommendations;""")

minnieRecs = """
SELECT songs.title, songs.artist
FROM Songs
JOIN Recommendations on Songs.song_id = Recommendations.song_id
WHERE Recommendations.user_id = (select user_id from Users where name = 'Minnie')
"""

runSql('Minnie Recommendations', minnieRecs)

user_id,song_id,recommendation_id,recommendation_time


title,artist


## [Optional] Python code for computing song similarity

In [ ]:
from collections import defaultdict
from datetime import datetime
import random

def compute_song_similarity(listens):
  song_similarity = defaultdict(int)
  common_users = defaultdict(set)

  # Compute song similarity and common users using defaultdict and set
  for listen1 in listens:
    user_id1 = listen1["user_id"]
    song_id1 = listen1["song_id"]

    for listen2 in listens:
      user_id2 = listen2["user_id"]
      song_id2 = listen2["song_id"]

      if user_id1 == user_id2 or song_id1 == song_id2:
        continue

      common_users[(song_id1, song_id2)].add(user_id1)
      song_similarity[(song_id1, song_id2)] += 1

  song_similarity_filtered = {
    (song_id1, song_id2): count
    for (song_id1, song_id2), count in song_similarity.items()
    if count > 1
  }
  return song_similarity_filtered, common_users

def compute_recommendations(listens, song_similarity, common_users):
  recommendations = []
  user_songs = defaultdict(set)

  # Compute recommendations for each user
  for listen in listens:
    user_id = listen["user_id"]
    song_id = listen["song_id"]

    user_songs[user_id].add(song_id)

  for (song_id1, song_id2), count in song_similarity.items():
    for user_id in common_users[(song_id1, song_id2)]:
      if song_id2 not in user_songs[user_id]:
        recommendations.append({
          "user_id": user_id,
          "song_id": song_id2,
          "recommendation_id": random.randint(1, 10000),
          "recommendation_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

  # Print the recommendations
  for recommendation in recommendations:
    print(f"User ID: {recommendation['user_id']}, Song ID: \
          {recommendation['song_id']}, \
          Recommendation ID: {recommendation['recommendation_id']}, \
          Recommendation Time: {recommendation['recommendation_time']}")

  return recommendations

def get_recommendations_for_user(user_name, songs, recommendations, users):
  user_id = next(user["user_id"] for user in users if user["name"] == user_name)

  recs = [
    song["song_id"]
    for rec in recommendations
    if rec["user_id"] == user_id
    for song in songs
    if song["song_id"] == rec["song_id"]
  ]

  recommendations = [
    {"title": song["title"], "artist": song["artist"]}
    for song in songs
    if song["song_id"] in recs
  ]
  # Print Minnie's recommendations
  for song in recommendations:
    print(f"Title: {song['title']}, Artist: {song['artist']}")
  return recommendations

song_similarity, common_users = compute_song_similarity(listens)
recommendations = compute_recommendations(listens, song_similarity, common_users)
get_recommendations_for_user('Minnie', songs, recommendations, users)


User ID: 1, Song ID:           7,           Recommendation ID: 675,           Recommendation Time: 2025-12-31 20:51:25
User ID: 3, Song ID:           7,           Recommendation ID: 4663,           Recommendation Time: 2025-12-31 20:51:25
User ID: 1, Song ID:           8,           Recommendation ID: 4352,           Recommendation Time: 2025-12-31 20:51:25
User ID: 3, Song ID:           8,           Recommendation ID: 9234,           Recommendation Time: 2025-12-31 20:51:25
User ID: 1, Song ID:           7,           Recommendation ID: 8124,           Recommendation Time: 2025-12-31 20:51:25
User ID: 3, Song ID:           7,           Recommendation ID: 6301,           Recommendation Time: 2025-12-31 20:51:25
User ID: 1, Song ID:           8,           Recommendation ID: 822,           Recommendation Time: 2025-12-31 20:51:25
User ID: 3, Song ID:           8,           Recommendation ID: 8541,           Recommendation Time: 2025-12-31 20:51:25
User ID: 2, Song ID:           1,         

[{'title': 'Evermore', 'artist': 'Taylor Swift'},
 {'title': 'Yesterday', 'artist': 'Beatles'}]

# Section 3: Compute Song Ratings (Intermediate+)
**Basics of Window Functions**

In [ ]:
""" Goal: Learn basic forms of Window Functions
- See how Partion+OVER are different than GROUP BYs"""

qry_group = """
  -- using GROUP BY
  SELECT user_id, song_id, AVG(rating) as avg_rating, SUM(rating) as ratings_sum
  FROM listens
  GROUP BY user_id, song_id
"""
qry_window = """
  WITH song_ratings AS (
      SELECT user_id, song_id, rating,
            SUM(rating) OVER (PARTITION BY user_id) as ratings_sum
      FROM listens
  )
  -- using window function
  SELECT user_id, song_id, rating, ratings_sum, rating/ratings_sum as ratio
  FROM song_ratings
"""

runSql('Group by Query', qry_group)
runSql('Window Query', qry_window)



user_id,song_id,avg_rating,ratings_sum
1,1,4.5,4.5
2,2,4.7,4.7
3,1,2.9,2.9
1,6,3.9,3.9
3,2,4.9,4.9
2,8,3.9,3.9
1,2,4.2,4.2
3,6,NaN,NaN
2,7,4.6,4.6


user_id,song_id,rating,ratings_sum,ratio
1,1,4.5,12.6,0.357143
1,2,4.2,12.6,0.333333
1,6,3.9,12.6,0.309524
2,2,4.7,13.2,0.356061
2,7,4.6,13.2,0.348485
2,8,3.9,13.2,0.295455
3,1,2.9,7.8,0.371795
3,2,4.9,7.8,0.628205
3,6,NaN,7.8,NaN


## [Optional] Equivalent python for Song Ratings with window vs group by function

In [ ]:
import math
def compute_song_ratings_window(listens):
  ratings_sum = {}

  # Compute SUM(rating) by PARTITION user_ids
  for listen in listens:
    user_id = listen["user_id"]
    rating = listen["rating"]

    if rating is not None:
      ratings_sum[user_id] = ratings_sum.get(user_id, 0) + rating

  results = []

  # Compute ratio using the ratings_sum values
  for listen in listens:
    user_id = listen["user_id"]
    song_id = listen["song_id"]
    rating = listen["rating"]

    ratio = math.nan if rating is None else rating / ratings_sum[user_id]
    results.append({"user_id": user_id, "song_id": song_id, "rating": rating,
                    "ratings_sum": ratings_sum[user_id], "ratio": ratio})

  # Print the results
  for result in results:
    print(f"User ID: {result['user_id']}, Song ID: {result['song_id']}, \
          Rating: {result['rating']}, Ratings Sum: {result['ratings_sum']}, \
          Ratio: {result['ratio']}")
compute_song_ratings_window(listens)

User ID: 1, Song ID: 1,           Rating: 4.5, Ratings Sum: 12.6,           Ratio: 0.35714285714285715
User ID: 1, Song ID: 2,           Rating: 4.2, Ratings Sum: 12.6,           Ratio: 0.33333333333333337
User ID: 1, Song ID: 6,           Rating: 3.9, Ratings Sum: 12.6,           Ratio: 0.30952380952380953
User ID: 2, Song ID: 2,           Rating: 4.7, Ratings Sum: 13.200000000000001,           Ratio: 0.356060606060606
User ID: 2, Song ID: 7,           Rating: 4.6, Ratings Sum: 13.200000000000001,           Ratio: 0.34848484848484845
User ID: 2, Song ID: 8,           Rating: 3.9, Ratings Sum: 13.200000000000001,           Ratio: 0.2954545454545454
User ID: 3, Song ID: 1,           Rating: 2.9, Ratings Sum: 7.800000000000001,           Ratio: 0.37179487179487175
User ID: 3, Song ID: 2,           Rating: 4.9, Ratings Sum: 7.800000000000001,           Ratio: 0.6282051282051282
User ID: 3, Song ID: 6,           Rating: None, Ratings Sum: 7.800000000000001,           Ratio: nan


In [ ]:
import math
from collections import defaultdict

def compute_song_ratings_group_by(listens):
  ratings_sum = defaultdict(float)
  ratings_count = defaultdict(int)

  # Compute ratings_sum and ratings_count using defaultdict
  for listen in listens:
    user_id = listen["user_id"]
    rating = listen["rating"]

    if rating is not None:
      ratings_sum[(user_id, listen["song_id"])] += rating
      ratings_count[(user_id, listen["song_id"])] += 1

  results = []

  # Compute avg_rating using ratings_sum and ratings_count
  for listen in listens:
    user_id = listen["user_id"]
    song_id = listen["song_id"]
    rating = listen["rating"]

    avg_rating = (
      ratings_sum[(user_id, song_id)] / ratings_count[(user_id, song_id)]
        if ratings_count[(user_id, song_id)] > 0
          else math.nan
    )
    results.append({
      "user_id": user_id,
      "song_id": song_id,
      "avg_rating": avg_rating,
      "ratings_sum": ratings_sum[(user_id, song_id)]
    })
  # Print the results
  for result in results:
    print(f"User ID: {result['user_id']}, Song ID: {result['song_id']}, \
            Average Rating: {result['avg_rating']}, \
            Ratings Sum: {result['ratings_sum']}")
compute_song_ratings_group_by(listens)

User ID: 1, Song ID: 1,             Average Rating: 4.5,             Ratings Sum: 4.5
User ID: 1, Song ID: 2,             Average Rating: 4.2,             Ratings Sum: 4.2
User ID: 1, Song ID: 6,             Average Rating: 3.9,             Ratings Sum: 3.9
User ID: 2, Song ID: 2,             Average Rating: 4.7,             Ratings Sum: 4.7
User ID: 2, Song ID: 7,             Average Rating: 4.6,             Ratings Sum: 4.6
User ID: 2, Song ID: 8,             Average Rating: 3.9,             Ratings Sum: 3.9
User ID: 3, Song ID: 1,             Average Rating: 2.9,             Ratings Sum: 2.9
User ID: 3, Song ID: 2,             Average Rating: 4.9,             Ratings Sum: 4.9
User ID: 3, Song ID: 6,             Average Rating: nan,             Ratings Sum: 0.0


# Section 4: Experiment with LLMs for text to SQL

LLMs can convert natural language to SQL. Useful for learning and speed.

**Pitfalls**: Queries may be syntactically correct but logically wrong. Edge cases and subtle JOINs trip them up.

Below, we look at two LLM-generated queries and debug them step by step.


## Prompt1: Find users who don't listen to Taylor Swift songs

In [ ]:
qryTaylorNoFan = """
SELECT u.user_id, u.name
FROM Users u
LEFT JOIN Listens l ON u.user_id = l.user_id
LEFT JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist != 'Taylor Swift' OR s.artist IS NULL
GROUP BY u.user_id, u.name
"""
runSql('Taylor No Fan', qryTaylorNoFan)

##################
### Below are the component parts and debug tables for the above query.
qryTaylorNoFan1a = """
SELECT u.user_id, u.name, l.song_id
FROM Users u
LEFT JOIN Listens l ON u.user_id = l.user_id
"""
runSql('Taylor No Fan 1a', qryTaylorNoFan1a)

### uncomment if you want to see debug tracing
qryTaylorNoFan1b = """
SELECT u.user_id, u.name, s.title, s.artist
FROM Users u
LEFT JOIN Listens l ON u.user_id = l.user_id
LEFT JOIN Songs s ON l.song_id = s.song_id
"""
runSql('Taylor No Fan 1b', qryTaylorNoFan1b)

### uncomment if you want to see debug tracing
qryTaylorNoFan1c = """
SELECT u.user_id, u.name, s.title, s.artist
FROM Users u
LEFT JOIN Listens l ON u.user_id = l.user_id
LEFT JOIN Songs s ON l.song_id = s.song_id
WHERE s.artist != 'Taylor Swift' OR s.artist IS NULL
"""
runSql('Taylor NoFan 1c', qryTaylorNoFan1c)




user_id,name
3,Daffy
4,Pluto
2,Minnie
1,Mickey


user_id,name,song_id
1,Mickey,1.0
1,Mickey,2.0
1,Mickey,6.0
2,Minnie,2.0
2,Minnie,7.0
2,Minnie,8.0
3,Daffy,1.0
3,Daffy,2.0
3,Daffy,6.0
4,Pluto,NaN


user_id,name,title,artist
1,Mickey,Evermore,Taylor Swift
1,Mickey,Willow,Taylor Swift
1,Mickey,Yesterday,Beatles
2,Minnie,Willow,Taylor Swift
2,Minnie,Yellow Submarine,Beatles
2,Minnie,Hey Jude,Beatles
3,Daffy,Evermore,Taylor Swift
3,Daffy,Willow,Taylor Swift
3,Daffy,Yesterday,Beatles
4,Pluto,None,None


user_id,name,title,artist
1,Mickey,Yesterday,Beatles
2,Minnie,Yellow Submarine,Beatles
2,Minnie,Hey Jude,Beatles
3,Daffy,Yesterday,Beatles
4,Pluto,None,None


In [ ]:
qryTaylorNoFanv2 = """
SELECT u.user_id, u.name
FROM Users u
WHERE u.user_id NOT IN (
    SELECT DISTINCT l.user_id
    FROM Listens l
    JOIN Songs s ON l.song_id = s.song_id
    WHERE s.artist = 'Taylor Swift'
)"""
runSql('Taylor NoFan', qryTaylorNoFanv2)

##################
### Below are the component parts and debug tables for the above query.
### uncomment if you want to see debug tracing
qryTaylorNoFanv2a = """
    SELECT DISTINCT l.user_id
    FROM Listens l
    JOIN Songs s ON l.song_id = s.song_id
    WHERE s.artist = 'Taylor Swift'
    """
runSql('Taylor NoFan v2a', qryTaylorNoFanv2a)


user_id,name
4,Pluto


user_id
1
2
3


## Prompt2: Find top3 songs for each genre

In [ ]:
qryTop3Songs = """
WITH ArtistAvgRating AS (
    SELECT artist, AVG(rating) as artist_avg_rating
    FROM Songs
    JOIN Listens ON Songs.song_id = Listens.song_id
    GROUP BY artist
),
SongRankings AS (
    SELECT
        s.genre,
        s.title,
        s.artist,
        s.song_id,
        AVG(l.rating) as avg_song_rating,
        a.artist_avg_rating,
        ROW_NUMBER() OVER (PARTITION BY s.genre ORDER BY AVG(l.rating) DESC) as rank
    FROM Songs s
    JOIN Listens l ON s.song_id = l.song_id
    JOIN ArtistAvgRating a ON s.artist = a.artist
    GROUP BY s.genre, s.title, s.song_id, s.artist, a.artist_avg_rating
)
--SELECT * from ArtistAvgRating;
SELECT * from SongRankings;
--SELECT genre, title, artist, avg_song_rating, artist_avg_rating, rank
--FROM SongRankings
--WHERE rank <= 3
--ORDER BY genre, rank
"""

####
# To see debug tables for the CTEs, uncomment Line# 21 or 22. And comment 23-26.
runSql('Top 3 songs per genre', qryTop3Songs)



genre,title,artist,song_id,avg_song_rating,artist_avg_rating,rank
Classic,Yellow Submarine,Beatles,7,4.6,4.133333,1
Classic,Hey Jude,Beatles,8,3.9,4.133333,2
Classic,Yesterday,Beatles,6,3.9,4.133333,3
Pop,Willow,Taylor Swift,2,4.6,4.240000,1
Pop,Evermore,Taylor Swift,1,3.7,4.240000,2


In [ ]:
"""ROW_NUMBER: assigns unique ranks (1,2,3) even with ties."""

qryTop3_RowNumber = """
WITH ArtistAvgRating AS (
    SELECT artist, AVG(rating) as artist_avg_rating
    FROM Songs
    LEFT JOIN Listens ON Songs.song_id = Listens.song_id
    GROUP BY artist
),
SongRankings AS (
    SELECT
        s.genre,
        s.title,
        s.artist,
        s.song_id,
        AVG(l.rating) as avg_song_rating,
        a.artist_avg_rating,
        ROW_NUMBER() OVER (PARTITION BY s.genre ORDER BY AVG(l.rating) DESC) as rank
        -- We see an example of ROW_NUMBER vs RANK(). Here it's the same because
        -- the AVG ratings are sorted.
    FROM Songs s
    LEFT JOIN Listens l ON s.song_id = l.song_id
    LEFT JOIN ArtistAvgRating a ON s.artist = a.artist
    GROUP BY s.genre, s.title, s.song_id, s.artist, a.artist_avg_rating
)
--SELECT * from ArtistAvgRating;
--SELECT * from SongRankings;
SELECT genre, title, artist, song_id, avg_song_rating, artist_avg_rating, rank
FROM SongRankings
WHERE rank <= 3
ORDER BY genre, rank
"""
runSql('Top 3 songs per genre (v2) with ROW_NUMBER', qryTop3_RowNumber)


genre,title,artist,song_id,avg_song_rating,artist_avg_rating,rank
Classic,Yellow Submarine,Beatles,7,4.6,4.133333,1
Classic,Hey Jude,Beatles,8,3.9,4.133333,2
Classic,Yesterday,Beatles,6,3.9,4.133333,3
Pop,Willow,Taylor Swift,2,4.6,4.240000,1
Pop,Evermore,Taylor Swift,1,3.7,4.240000,2
Rock,Photograph,Ed Sheeran,4,NaN,NaN,1
Rock,Shivers,Ed Sheeran,5,NaN,NaN,2
Rock,Bad Blood,Taylor Swift,9,NaN,4.240000,3
None,DJ Mix,DJ,10,NaN,NaN,1


In [ ]:
"""RANK: assigns same rank to ties, then skips (1,1,3 instead of 1,2,3)."""

qryTop3_Rank = """
WITH ArtistAvgRating AS (
    SELECT artist, AVG(rating) as artist_avg_rating
    FROM Songs
    LEFT JOIN Listens ON Songs.song_id = Listens.song_id
    GROUP BY artist
),
SongRankings AS (
    SELECT
        s.genre,
        s.title,
        s.artist,
        AVG(l.rating) as avg_song_rating,
        a.artist_avg_rating,
        RANK() OVER (PARTITION BY s.genre ORDER BY AVG(l.rating) DESC) as rank
        -- We see an example of ROW_NUMBER vs RANK(). Here it's the same because
        -- the AVG ratings are sorted.
    FROM Songs s
    LEFT JOIN Listens l ON s.song_id = l.song_id
    LEFT JOIN ArtistAvgRating a ON s.artist = a.artist
    GROUP BY s.genre, s.title, s.artist, a.artist_avg_rating
)
--SELECT * from ArtistAvgRating;
--SELECT * from SongRankings;
SELECT genre, title, artist, avg_song_rating, artist_avg_rating, rank
FROM SongRankings
WHERE rank <= 3
ORDER BY genre, rank
"""
runSql('Top 3 songs per genre (v2) with RANK', qryTop3_Rank)


genre,title,artist,avg_song_rating,artist_avg_rating,rank
Classic,Yellow Submarine,Beatles,4.6,4.133333,1
Classic,Hey Jude,Beatles,3.9,4.133333,2
Classic,Yesterday,Beatles,3.9,4.133333,2
Pop,Willow,Taylor Swift,4.6,4.240000,1
Pop,Evermore,Taylor Swift,3.7,4.240000,2
Rock,Photograph,Ed Sheeran,NaN,NaN,1
Rock,Shivers,Ed Sheeran,NaN,NaN,1
Rock,Shape of You,Ed Sheeran,NaN,NaN,1
Rock,Bad Blood,Taylor Swift,NaN,4.240000,1
None,DJ Mix,DJ,NaN,NaN,1


# Section 5: Extended dataset for the practice problems

Adds Songs 11-18, more Listens (from users 1-3), and a fixed set of Recommendations for Mickey, on top of the canonical seed. The canonical Users 1-4 / Songs 1-10 / Listens 1-9 are untouched, and Pluto stays the only user with no listens. Run this once before the practice problems (it is safe to re-run). It does not change any cell above it.

In [ ]:
# --- Section 5: extended dataset for the practice problems (additive, safe to re-run) ---
# Only touches song_id >= 11, listen_id >= 10, and Recommendations. Canonical rows are kept.
runSql("Reset practice rows",
       "DELETE FROM Recommendations; DELETE FROM Listens WHERE listen_id >= 10; DELETE FROM Songs WHERE song_id >= 11;")

runSql("Add practice Songs (11-18)", """
INSERT INTO Songs (song_id, title, artist, genre) VALUES
  (11, 'Anti-Hero', 'Taylor Swift', 'Pop'),
  (12, 'Cruel Summer', 'Taylor Swift', 'Pop'),
  (13, 'Perfect', 'Ed Sheeran', 'Rock'),
  (14, 'Castle on the Hill', 'Ed Sheeran', 'Rock'),
  (15, 'Let It Be', 'Beatles', 'Classic'),
  (16, 'Come Together', 'Beatles', 'Classic'),
  (17, 'Hello', 'Adele', 'Pop'),
  (18, 'Rolling in the Deep', 'Adele', 'Pop');
""")

runSql("Add practice Listens", """
INSERT INTO Listens (listen_id, user_id, song_id, rating) VALUES
  (10, 2, 1, 4.0),
  (11, 3, 3, 4.4),
  (12, 1, 3, 3.7),
  (13, 2, 3, 4.8),
  (14, 3, 4, 4.1),
  (15, 1, 4, 4.0),
  (16, 2, 5, 4.4),
  (17, 3, 9, 3.7),
  (18, 1, 9, 4.8),
  (19, 2, 11, 4.1),
  (20, 3, 11, 4.0),
  (21, 1, 11, 4.4),
  (22, 2, 11, 3.7),
  (23, 3, 11, 4.8),
  (24, 1, 12, 4.1),
  (25, 2, 13, 4.0),
  (26, 3, 13, 4.4),
  (27, 1, 13, 3.7),
  (28, 2, 13, 4.8),
  (29, 3, 14, 4.1),
  (30, 1, 14, 4.0),
  (31, 2, 15, 4.4),
  (32, 3, 15, 3.7),
  (33, 1, 15, 4.8),
  (34, 2, 16, 4.1),
  (35, 3, 16, 4.0),
  (36, 1, 17, 4.4),
  (37, 2, 17, 3.7),
  (38, 3, 17, 4.8),
  (39, 1, 17, 4.1),
  (40, 2, 18, 4.0);
""")

runSql("Set Mickey's Recommendations", """
INSERT INTO Recommendations (user_id, song_id) VALUES
  (1, 3),
  (1, 9),
  (1, 11),
  (1, 17);
""")

runSql("Verify", "SELECT (SELECT COUNT(*) FROM Songs) AS songs, (SELECT COUNT(*) FROM Listens) AS listens, (SELECT COUNT(*) FROM Recommendations) AS recs;")